In [ ]:
# !pip install -U gensim

In [2]:
import pandas as pd
import numpy as np
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity
pd.set_option('display.max_colwidth', None)

c:\Users\Noman\anaconda3\Lib\site-packages\paramiko\pkey.py:82: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "cipher": algorithms.TripleDES,
c:\Users\Noman\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.Blowfish and will be removed from this module in 45.0.0.
  "class": algorithms.Blowfish,
c:\Users\Noman\anaconda3\Lib\site-packages\paramiko\transport.py:243: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from this module in 48.0.0.
  "class": algorithms.TripleDES,


In [3]:
df = pd.read_csv('products.csv')

In [4]:
df.head(2)

,Product Name,Group,Item,Availability,Color,Link,Size,Price,Fit,Fabric,user_query,Hashtags,Image
0,Black Genuine Leather Auto Lock Belt,Men,Belt,In Stock,Black,https://www.aarong.com/catalog/product/view/id/2650374/s/black-genuine-leather-auto-lock-belt-10d248715002/category/60/,36,"Tk 1,323.81",Not Applicable,Not Applicable,"Something trendy for Men, maybe a Black Accessories?","['black belt', 'belt', 'size 36 black belt', 'black men belt', 'black', '36 black genuine belt', 'men 36', 'black black', 'black belt for men', 'belt belt 36', 'black men size 36 belt']","['https://www.productsamples.com/wp-content/uploads/2021/10/il-box.png', 'https://www.productsamples.com/wp-content/uploads/2021/10/il-box.png', 'https://www.productsamples.com/wp-content/uploads/2021/10/il-box.png']"
1,Black Genuine Leather Auto Lock Belt,Men,Belt,In Stock,Black,https://www.aarong.com/catalog/product/view/id/2650374/s/black-genuine-leather-auto-lock-belt-10d248715002/category/60/,42,"Tk 1,323.81",Not Applicable,Not Applicable,Do you have any Accessoriess for Men in Black?,"['black belt', 'belt', 'size 42 black belt', 'black men belt', 'black', '42 black genuine belt', 'men 42', 'black black', 'black belt for men', 'belt belt 42', 'black men size 42 belt']","['https://www.productsamples.com/wp-content/uploads/2021/10/il-box.png', 'https://www.productsamples.com/wp-content/uploads/2021/10/il-box.png', 'https://www.productsamples.com/wp-content/uploads/2021/10/il-box.png']"


In [5]:
df.shape

(8147, 13)

In [6]:
# Product Name,Group , Item , Color , Fit , Fabric

new_df = df[['Product Name','Group','Item','Color','Fit','Fabric', 'Link']]
new_df.head(3)

,Product Name,Group,Item,Color,Fit,Fabric,Link
0,Black Genuine Leather Auto Lock Belt,Men,Belt,Black,Not Applicable,Not Applicable,https://www.aarong.com/catalog/product/view/id/2650374/s/black-genuine-leather-auto-lock-belt-10d248715002/category/60/
1,Black Genuine Leather Auto Lock Belt,Men,Belt,Black,Not Applicable,Not Applicable,https://www.aarong.com/catalog/product/view/id/2650374/s/black-genuine-leather-auto-lock-belt-10d248715002/category/60/
2,Black Genuine Leather Auto Lock Belt,Men,Belt,Black,Not Applicable,Not Applicable,https://www.aarong.com/catalog/product/view/id/2650374/s/black-genuine-leather-auto-lock-belt-10d248715002/category/60/


In [7]:
new_df.isnull().sum()

Product Name    0
Group           0
Item            0
Color           0
Fit             0
Fabric          0
Link            0
dtype: int64

In [8]:
new_df.duplicated().sum()

5000

In [9]:
new_df.drop_duplicates(inplace=True)

C:\Users\Noman\AppData\Local\Temp\ipykernel_13804\2373844780.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df.drop_duplicates(inplace=True)


In [10]:
new_df.reset_index(drop=True, inplace=True)

In [11]:
new_df.shape

(3147, 7)

In [12]:
# Replace 'Accessories' in 'Item' with the last word of 'Product Name', with special cases
def update_item(row):
    if row['Item'] == 'Accessories':
        # Special cases for Jewellery Organiser/Storage
        if row['Product Name'] in ['Black Fabric Jewellery Organiser/Storage',
                                  'Maroon Fabric Jewellery Organiser/Storage']:
            return 'Jewellery'
        # General case: use the last word of Product Name
        return row['Product Name'].split()[-1]
    return row['Item']

new_df['Item'] = new_df.apply(update_item, axis=1)
new_df.head()

C:\Users\Noman\AppData\Local\Temp\ipykernel_13804\3686094292.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['Item'] = new_df.apply(update_item, axis=1)


,Product Name,Group,Item,Color,Fit,Fabric,Link
0,Black Genuine Leather Auto Lock Belt,Men,Belt,Black,Not Applicable,Not Applicable,https://www.aarong.com/catalog/product/view/id/2650374/s/black-genuine-leather-auto-lock-belt-10d248715002/category/60/
1,Black Genuine Leather Bag,Men,Bag,Black,Not Applicable,Not Applicable,https://www.aarong.com/men/accessories/black-genuine-leather-bag-0870000107580.html
2,White Check Fabric Cross Body Bag,Men,Bag,White,Not Applicable,Not Applicable,https://www.aarong.com/catalog/product/view/id/943561/s/white-check-fabric-cross-body-bag/category/60/
3,Red Printed Genuine Leather Taaga Man Wallet,Men,Wallet,Red,Not Applicable,Not Applicable,https://www.aarong.com/men/accessories/red-printed-genuine-leather-taaga-man-wallet-1210000002333.html
4,Chocolate Genuine Leather Wallet,Men,Wallet,Brown,Not Applicable,Not Applicable,https://www.aarong.com/catalog/product/view/id/712706/s/chocolate-genuine-leather-wallet-0870000104531/category/60/


In [13]:
new_df["Fit"] = new_df["Fit"].apply(lambda x: x.replace(" ", ""))

C:\Users\Noman\AppData\Local\Temp\ipykernel_13804\926375751.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df["Fit"] = new_df["Fit"].apply(lambda x: x.replace(" ", ""))


In [14]:
new_df["Fit"]

0       NotApplicable
1       NotApplicable
2       NotApplicable
3       NotApplicable
4       NotApplicable
            ...      
3142           A-Line
3143        FrontOpen
3144           A-Line
3145           Flared
3146          Pleated
Name: Fit, Length: 3147, dtype: object

In [15]:
new_df["Item"] = new_df["Item"].apply(lambda x: x.replace(" ", ""))
new_df["Color"] = new_df["Color"].apply(lambda x: x.replace(" ", ""))
new_df["Fabric"] = new_df["Fabric"].apply(lambda x: x.replace(" ", ""))

C:\Users\Noman\AppData\Local\Temp\ipykernel_13804\4172857338.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df["Item"] = new_df["Item"].apply(lambda x: x.replace(" ", ""))
C:\Users\Noman\AppData\Local\Temp\ipykernel_13804\4172857338.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df["Color"] = new_df["Color"].apply(lambda x: x.replace(" ", ""))
C:\Users\Noman\AppData\Local\Temp\ipykernel_13804\4172857338.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a 

In [16]:
# new_df["Item1"] =

In [17]:
new_df.head(2)

,Product Name,Group,Item,Color,Fit,Fabric,Link
0,Black Genuine Leather Auto Lock Belt,Men,Belt,Black,NotApplicable,NotApplicable,https://www.aarong.com/catalog/product/view/id/2650374/s/black-genuine-leather-auto-lock-belt-10d248715002/category/60/
1,Black Genuine Leather Bag,Men,Bag,Black,NotApplicable,NotApplicable,https://www.aarong.com/men/accessories/black-genuine-leather-bag-0870000107580.html


In [18]:
# Combine attributes into a space-separated tags column
# Create tags
new_df['tags'] = new_df['Group'].str.lower() + ' ' + new_df['Group'].str.lower() + ' ' + new_df['Item'].str.lower() + ' ' + new_df['Item'].str.lower() + ' ' + new_df['Color'].str.lower() + ' ' + new_df['Fit'].str.lower() + ' ' + new_df['Fabric'].str.lower()

# Now drop the original columns if you want
new_df = new_df.drop(columns=['Group', 'Item', 'Color', 'Fit', 'Fabric'])

# View first few rows to check
new_df.head()

C:\Users\Noman\AppData\Local\Temp\ipykernel_13804\1295588410.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['Group'].str.lower() + ' ' + new_df['Group'].str.lower() + ' ' + new_df['Item'].str.lower() + ' ' + new_df['Item'].str.lower() + ' ' + new_df['Color'].str.lower() + ' ' + new_df['Fit'].str.lower() + ' ' + new_df['Fabric'].str.lower()


,Product Name,Link,tags
0,Black Genuine Leather Auto Lock Belt,https://www.aarong.com/catalog/product/view/id/2650374/s/black-genuine-leather-auto-lock-belt-10d248715002/category/60/,men men belt belt black notapplicable notapplicable
1,Black Genuine Leather Bag,https://www.aarong.com/men/accessories/black-genuine-leather-bag-0870000107580.html,men men bag bag black notapplicable notapplicable
2,White Check Fabric Cross Body Bag,https://www.aarong.com/catalog/product/view/id/943561/s/white-check-fabric-cross-body-bag/category/60/,men men bag bag white notapplicable notapplicable
3,Red Printed Genuine Leather Taaga Man Wallet,https://www.aarong.com/men/accessories/red-printed-genuine-leather-taaga-man-wallet-1210000002333.html,men men wallet wallet red notapplicable notapplicable
4,Chocolate Genuine Leather Wallet,https://www.aarong.com/catalog/product/view/id/712706/s/chocolate-genuine-leather-wallet-0870000104531/category/60/,men men wallet wallet brown notapplicable notapplicable


In [19]:
new_df["tags"][0]

'men men belt belt black notapplicable notapplicable'

In [20]:
new_df['tags']=new_df['tags'].apply(lambda x:x.lower())

In [21]:
new_df.head(3)

,Product Name,Link,tags
0,Black Genuine Leather Auto Lock Belt,https://www.aarong.com/catalog/product/view/id/2650374/s/black-genuine-leather-auto-lock-belt-10d248715002/category/60/,men men belt belt black notapplicable notapplicable
1,Black Genuine Leather Bag,https://www.aarong.com/men/accessories/black-genuine-leather-bag-0870000107580.html,men men bag bag black notapplicable notapplicable
2,White Check Fabric Cross Body Bag,https://www.aarong.com/catalog/product/view/id/943561/s/white-check-fabric-cross-body-bag/category/60/,men men bag bag white notapplicable notapplicable


In [22]:
# Remove "not applicable" from tags
new_df['tags'] = new_df['tags'].str.replace('notapplicable', '', regex=False)

# Strip extra spaces afterwards
new_df['tags'] = new_df['tags'].str.split().str.join(' ')

In [23]:
new_df.tail()

,Product Name,Link,tags
3142,Olive Striped Mixed Cotton Taaga Top,https://www.aarong.com/catalog/product/view/id/2977748/s/olive-striped-mixed-cotton-taaga-top-22e254150278/category/3802/,women women tops tops olive a-line mixedcotton
3143,White Embroidered Cotton Taaga Top,https://www.aarong.com/catalog/product/view/id/1974691/s/white-embroidered-cotton-taaga-top-22z234150070/category/3802/,women women tops tops white frontopen cotton
3144,Brown Polyester Camisole,https://www.aarong.com/catalog/product/view/id/2927461/s/brown-polyester-camisole-22sb244161001/category/3802/,women women tops tops brown a-line polyester
3145,Hot Pink Printed and Embroidered Tencel-Vortex Taaga Dressy Tops,https://www.aarong.com/catalog/product/view/id/2976731/s/hot-pink-printed-and-embroidered-tencel-vortex-taaga-dressy-tops-22e254150161/category/3802/,women women tops tops pink flared mixedcotton
3146,White Textured Viscose-Cotton Taaga Top,https://www.aarong.com/catalog/product/view/id/1942297/s/white-textured-viscose-cotton-taaga-top-22e224150039/category/3802/,women women tops tops white pleated viscott(viscose&cotton)


# **Vectorization**

In [24]:
# Prepare data for Word2Vec
sentences = [row.split() for row in new_df['tags']]
word2vec_model = Word2Vec(sentences, vector_size=100, window=5, min_count=1, workers=4)

In [25]:
# Function to get average Word2Vec vector for a sentence
def get_avg_word2vec(tags):
    words = tags.split()
    vec = np.zeros(100)
    count = 0
    for word in words:
        if word in word2vec_model.wv:
            vec += word2vec_model.wv[word]
            count += 1
    if count > 0:
        vec /= count
    return vec

In [26]:
# Create vectors
vectors = np.array([get_avg_word2vec(tags) for tags in new_df['tags']])
vectors[0]

array([-6.44226976e-02,  3.27568585e-01, -1.33689246e-01,  1.83616930e-01,
        2.44406295e-02,  5.04420560e-02,  7.18270212e-02,  8.70872393e-02,
       -3.50575894e-03, -1.14332286e-01,  1.02451567e-01, -1.70538536e-01,
        1.95107406e-01,  7.25294784e-02,  1.66104750e-02,  1.51479796e-01,
        2.07940784e-01,  8.35436381e-02,  7.83896625e-02, -7.08622418e-02,
       -6.22239709e-04, -2.01107374e-01, -1.44783202e-01,  7.92140178e-02,
        1.97903913e-02,  1.07076992e-01, -1.87131682e-01,  4.23894830e-02,
        1.03929886e-02, -2.78275441e-02, -1.20969990e-02,  1.05563191e-01,
        2.81219184e-02, -3.33323503e-01, -1.23615909e-02,  2.22469079e-01,
        2.45194325e-01, -2.83403307e-02, -2.32334411e-01,  1.13309158e-01,
       -5.58132365e-02,  1.51660910e-01,  4.73670736e-03, -1.81507549e-01,
        1.31636006e-01, -9.67838421e-02,  7.05021217e-02, -2.56464295e-02,
        2.14453936e-02,  7.24620253e-02,  9.51651990e-02, -1.11733656e-04,
       -2.49326843e-01, -

# **Cosine Similarity**

In [27]:
# Calculate cosine similarity
similarity = cosine_similarity(vectors)

In [28]:
similarity[0]

array([1.        , 0.99947117, 0.99794769, ..., 0.90991755, 0.91130801,
       0.9268993 ])

In [29]:
# Recommendation function
def recommend(cloth):
  try:
      cloth_index = new_df[new_df['Product Name'] == cloth].index[0]
      distances = similarity[cloth_index]
      cloth_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:10]  # Get top 10 to filter later

      # Filter out recommendations with the same product name
      recommended = []
      for i in cloth_list:
          if new_df.iloc[i[0]]['Product Name'] != cloth:
              recommended.append(i)
          if len(recommended) >= 5:  # Stop when we have 5 valid recommendations
              break

      # Print recommendations
      if not recommended:
          print("No recommendations found with different product names.")
          return

      for i in recommended:
          print(new_df.iloc[i[0]]['Product Name'])
          print(new_df.iloc[i[0]]['Link'])

  except IndexError:
      print(f"Product '{cloth}' not found in the dataset.")
      return

  return

In [30]:
recommend('Olive Striped Mixed Cotton Taaga Top')

Purple Embroidered Viscose-Cotton Taaga Dressy Top
https://www.aarong.com/catalog/product/view/id/2527985/s/purple-embroidered-viscose-cotton-taaga-dressy-top-22s234150001/category/3802/
Lavender Embroidered Viscose-Cotton Taaga Casual Top
https://www.aarong.com/catalog/product/view/id/2854326/s/lavender-embroidered-viscose-cotton-taaga-casual-top-22e254150073/category/3802/
Black Printed Mixed Tencel Taaga Top
https://www.aarong.com/catalog/product/view/id/2985279/s/black-printed-mixed-tencel-taaga-top-22l254149006/category/3802/
Coffee Printed and Embroidered Viscose Taaga Top
https://www.aarong.com/catalog/product/view/id/2018596/s/coffee-printed-and-embroidered-viscose-taaga-top-22e234150309/category/3802/
Grey Polyester Taaga Tank Top
https://www.aarong.com/catalog/product/view/id/2925049/s/grey-polyester-taaga-tank-top-22u244161002/category/3802/


In [31]:
recommend('Green Printed and Textured Cotton Fatua')

Green Printed Cotton Fatua
https://www.aarong.com/catalog/product/view/id/2988741/s/green-printed-cotton-fatua-03e250805114/category/16/
Teal Printed Cotton Fatua
https://www.aarong.com/catalog/product/view/id/2975834/s/teal-printed-cotton-fatua-03e250805109/category/16/
Burnt Orange Printed Cotton Fatua
https://www.aarong.com/men/fatua/burnt-orange-printed-cotton-fatua-03v240805001.html
Coral Printed and Embroidered Cotton Fatua
https://www.aarong.com/catalog/product/view/id/2682030/s/coral-printed-and-embroidered-cotton-fatua-03z240806034/category/16/
Turquoise Embroidered Cotton Fatua
https://www.aarong.com/catalog/product/view/id/2576524/s/turquoise-embroidered-cotton-fatua-03o230806005/category/16/


In [32]:
recommend('Sand Brown/Green Striped Cotton Lungi')

Beige Striped Cotton Lungi
https://www.aarong.com/catalog/product/view/id/2371941/s/beige-striped-cotton-lungi-0090000093179/category/17/
Brown Check Cotton Lungi
https://www.aarong.com/catalog/product/view/id/2885865/s/brown-check-cotton-lungi-0090000093729/category/17/
Pink/Chocolate Check Cotton Lungi
https://www.aarong.com/catalog/product/view/id/2832282/s/pink-chocolate-check-cotton-lungi-0090000093639/category/17/
White/Brown Check Cotton Lungi
https://www.aarong.com/catalog/product/view/id/2812665/s/white-brown-check-cotton-lungi-0090000093650/category/17/
Brown Check Cotton Lungi
https://www.aarong.com/catalog/product/view/id/2339139/s/brown-check-cotton-lungi-0090000093162/category/17/


In [33]:
recommend('Midnight Blue Printed Viscose-Cotton Panjabi')

Blue Check and Embroidered Viscose-Cotton Panjabi
https://www.aarong.com/catalog/product/view/id/2828535/s/blue-check-and-embroidered-viscose-cotton-panjabi-15r250360288/category/12/
Blue Dyed and Embroidered Viscose-Cotton Panjabi
https://www.aarong.com/catalog/product/view/id/2705609/s/blue-dyed-and-embroidered-viscose-cotton-panjabi-15e240360737/category/12/
Rust Embroidered and Printed Viscose-Cotton Panjabi
https://www.aarong.com/catalog/product/view/id/2829558/s/rust-embroidered-and-printed-viscose-cotton-panjabi-15r250360063/category/12/
Blue Striped Viscose-Cotton Panjabi
https://www.aarong.com/catalog/product/view/id/2829534/s/blue-striped-viscose-cotton-panjabi-15e250360085/category/12/
White Printed Viscose-Cotton Panjabi
https://www.aarong.com/catalog/product/view/id/2812542/s/white-printed-viscose-cotton-panjabi-15r250360190/category/12/


In [34]:
recommend('Brown Genuine Leather Wallet')

Chocolate Leather Wallet
https://www.aarong.com/catalog/product/view/id/2337447/s/chocolate-leather-wallet-0870000106939/category/60/
Black/Wine Genuine Leather Wallet
https://www.aarong.com/catalog/product/view/id/2650272/s/black-wine-genuine-leather-wallet-0870000107813/category/60/
Black Textured Genuine Leather Wallet
https://www.aarong.com/men/accessories/black-textured-genuine-leather-wallet-0870000107793.html
Black Leather Wallet
https://www.aarong.com/men/accessories/black-leather-wallet-0870000108371.html
Black Leather Wallet
https://www.aarong.com/catalog/product/view/id/125603/s/10e178709007/category/60/


In [35]:
recommend('Brown Polyester Camisole')

Black Printed Mixed Tencel Taaga Top
https://www.aarong.com/catalog/product/view/id/2985279/s/black-printed-mixed-tencel-taaga-top-22l254149006/category/3802/
Coffee Printed and Embroidered Viscose Taaga Top
https://www.aarong.com/catalog/product/view/id/2018596/s/coffee-printed-and-embroidered-viscose-taaga-top-22e234150309/category/3802/
Orange Viscose-Cotton Taaga Top
https://www.aarong.com/catalog/product/view/id/1974688/s/orange-viscose-cotton-taaga-top-22e234150351/category/3802/
Brown Printed and Embroidered Mixed Viscose Taaga Top
https://www.aarong.com/catalog/product/view/id/2501018/s/brown-printed-and-embroidered-mixed-viscose-taaga-top-22f224150001/category/3802/
Olive Striped Mixed Cotton Taaga Top
https://www.aarong.com/catalog/product/view/id/2977748/s/olive-striped-mixed-cotton-taaga-top-22e254150278/category/3802/


In [36]:
recommend('Multicolour Printed Cotton T-Shirt')

Grey Printed Cotton T-Shirt
https://www.aarong.com/catalog/product/view/id/2705708/s/grey-printed-cotton-t-shirt-03gg241201016/category/176/
Grey/Green Tie-Dyed Cotton T-Shirt
https://www.aarong.com/men/t-shirts/grey-green-tie-dyed-cotton-t-shirt-03l241201008.html
Grey Printed Cotton T-Shirt
https://www.aarong.com/men/t-shirts/grey-printed-cotton-t-shirt-03g241201013.html
Grey Printed Cotton T-Shirt
https://www.aarong.com/catalog/product/view/id/2976683/s/grey-printed-cotton-t-shirt-03gg251201023/category/176/
Charcoal Grey Printed Cotton T-Shirt
https://www.aarong.com/catalog/product/view/id/2976680/s/charcoal-grey-printed-cotton-t-shirt-03z251201023/category/176/


In [37]:
recommend('Black Embroidered Genuine Leather Nagras')

Black Genuine Leather Slipper Sandal
https://www.aarong.com/men/shoes/black-genuine-leather-slipper-sandal-10z238731002.html
Black Genuine Leather Moccasin Shoes
https://www.aarong.com/men/shoes/black-genuine-leather-moccasin-shoes-10e248784003.html
Black Suede Leather Sandal
https://www.aarong.com/catalog/product/view/id/2102146/s/black-suede-leather-sandal-10d228731002/category/19/
Black Genuine Leather Thong Sandal
https://www.aarong.com/men/shoes/black-genuine-leather-thong-sandal-10e258731001.html
Black Genuine Leather Strap Sandals
https://www.aarong.com/men/shoes/black-genuine-leather-strap-sandals-10e228731014.html


In [38]:
recommend('Green Embroidered and Printed Handloom Viscose Ethnic Scarf')

Product 'Green Embroidered and Printed Handloom Viscose Ethnic Scarf' not found in the dataset.


In [39]:
recommend('Red Erri Embroidered Chosha Silk Purse')

Multicolour Embroidered Jute Purse
https://www.aarong.com/catalog/product/view/id/2362563/s/multicolour-embroidered-jute-purse-0940000096741/category/61/
Red Textured Genuine Leather Bag
https://www.aarong.com/catalog/product/view/id/2618356/s/red-textured-genuine-leather-bag-0870000107676/category/61/
Brick Red Genuine Leather Purse
https://www.aarong.com/catalog/product/view/id/209628/s/10g188704001/category/61/
Red Genuine Leather Wallet
https://www.aarong.com/catalog/product/view/id/160000/s/10r188704024-0d455f/category/61/
Red Velvet Party Bag
https://www.aarong.com/catalog/product/view/id/616630/s/red-velvet-party-bag/category/61/


In [40]:
recommend('Black Genuine Leather Belt')

Black Mesh Genuine Leather Belt
https://www.aarong.com/men/accessories/black-mesh-genuine-leather-belt-10e238715003.html
Black Embossed Genuine Leather Belt
https://www.aarong.com/catalog/product/view/id/547963/s/black-embossed-leather-belt-10lb188715003/category/60/
Black and Brown Embossed Genuine Leather Belt
https://www.aarong.com/catalog/product/view/id/968758/s/black-and-brown-embossed-genuine-leather-belt-10f198715002/category/60/
Chocolate Genuine Leather Belt
https://www.aarong.com/catalog/product/view/id/688356/s/chocolate-leather-belt-10e218715008/category/60/
Tan Genuine Leather Belt
https://www.aarong.com/men/accessories/tan-leather-belt-10et208715005.html


In [41]:
recommend('Brown Genuine Leather Strap Sandals')

Chocolate Genuine Leather Half Shoes
https://www.aarong.com/men/shoes/chocolate-genuine-leather-half-shoes-10j258731001.html
Chocolate Genuine Leather Half Moccasin Shoes
https://www.aarong.com/catalog/product/view/id/2574418/s/chocolate-genuine-leather-half-moccasin-shoes-10e248731002/category/19/
Brown Genuine Leather Thong Sandal
https://www.aarong.com/men/shoes/brown-genuine-leather-thong-sandal-10t258731001.html
Tan/Brown Genuine Leather Sandals
https://www.aarong.com/men/shoes/tan-brown-genuine-leather-sandals-10f238731003.html
Brown Leather Nagra
https://www.aarong.com/men/shoes/brown-leather-nagra-10eb198731028.html


In [42]:
recommend('Chocolate Genuine Leather Half Shoes')

Chocolate Genuine Leather Half Moccasin Shoes
https://www.aarong.com/catalog/product/view/id/2574418/s/chocolate-genuine-leather-half-moccasin-shoes-10e248731002/category/19/
Brown Genuine Leather Thong Sandal
https://www.aarong.com/men/shoes/brown-genuine-leather-thong-sandal-10t258731001.html
Tan/Brown Genuine Leather Sandals
https://www.aarong.com/men/shoes/tan-brown-genuine-leather-sandals-10f238731003.html
Brown Leather Nagra
https://www.aarong.com/men/shoes/brown-leather-nagra-10eb198731028.html
Brown Leather Nagra
https://www.aarong.com/catalog/product/view/id/339506/s/brown-leather-nagra/category/19/


In [43]:
new_df.iloc[211]

Product Name                                                         Brick Red Cotton Taaga Man Polo Shirt
Link            https://www.aarong.com/men/jackets/brick-red-cotton-taaga-man-polo-shirt-31r245706002.html
tags                                                         men men jackets jackets red regularfit cotton
Name: 211, dtype: object

In [44]:
new_df.iloc[8]

Product Name                                                                                        Brown Genuine Leather Wallet
Link            https://www.aarong.com/catalog/product/view/id/2843244/s/brown-genuine-leather-wallet-0870000108170/category/60/
tags                                                                                                 men men wallet wallet brown
Name: 8, dtype: object

In [45]:
new_df.tail()

,Product Name,Link,tags
3142,Olive Striped Mixed Cotton Taaga Top,https://www.aarong.com/catalog/product/view/id/2977748/s/olive-striped-mixed-cotton-taaga-top-22e254150278/category/3802/,women women tops tops olive a-line mixedcotton
3143,White Embroidered Cotton Taaga Top,https://www.aarong.com/catalog/product/view/id/1974691/s/white-embroidered-cotton-taaga-top-22z234150070/category/3802/,women women tops tops white frontopen cotton
3144,Brown Polyester Camisole,https://www.aarong.com/catalog/product/view/id/2927461/s/brown-polyester-camisole-22sb244161001/category/3802/,women women tops tops brown a-line polyester
3145,Hot Pink Printed and Embroidered Tencel-Vortex Taaga Dressy Tops,https://www.aarong.com/catalog/product/view/id/2976731/s/hot-pink-printed-and-embroidered-tencel-vortex-taaga-dressy-tops-22e254150161/category/3802/,women women tops tops pink flared mixedcotton
3146,White Textured Viscose-Cotton Taaga Top,https://www.aarong.com/catalog/product/view/id/1942297/s/white-textured-viscose-cotton-taaga-top-22e224150039/category/3802/,women women tops tops white pleated viscott(viscose&cotton)


In [46]:
# Regular expression pattern for the specific format
pattern = r"^https://www\.aarong\.com/men/shoes/.+-[0-9a-zA-Z]+\.html$"

# Filter rows that match the pattern
filtered_links = df[df['Link'].str.contains(pattern, regex=True, na=False)]

# Print the matching links
print(filtered_links['Link'])

3562             https://www.aarong.com/men/shoes/chocolate-genuine-leather-half-shoes-10j258731001.html
3563             https://www.aarong.com/men/shoes/chocolate-genuine-leather-half-shoes-10j258731001.html
3564              https://www.aarong.com/men/shoes/midnight-blue-suede-leather-sandals-10e238731005.html
3565              https://www.aarong.com/men/shoes/midnight-blue-suede-leather-sandals-10e238731005.html
3566              https://www.aarong.com/men/shoes/midnight-blue-suede-leather-sandals-10e238731005.html
                                                      ...                                               
3651               https://www.aarong.com/men/shoes/brown-genuine-leather-thong-sandal-10x248731006.html
3652               https://www.aarong.com/men/shoes/brown-genuine-leather-thong-sandal-10x248731006.html
3653    https://www.aarong.com/men/shoes/light-brown-genuine-leather-kolhapuri-sandals-10d248731002.html
3654                    https://www.aarong.com/men/shoe

In [47]:
# Save the Word2Vec model
word2vec_model.save("word2vec_model.model")

# Save the similarity matrix
np.save("similarity_matrix.npy", similarity)

# Save the processed DataFrame
new_df.to_pickle("processed_data.pkl")